# 从零实现 DiffPool：可微分簇分配与两层层次图分类

本 Notebook 只用 PyTorch 基础张量和 `nn.Module`，手写 dense GCN、assignment network、masked row softmax、$S^TX$、$S^TAS$、link-prediction auxiliary loss、entropy regularizer，以及两层 hierarchical DiffPool；不使用 PyG、DGL 或任何现成 GNN/池化层。

可执行合同覆盖：多图 padding、邻接对称性、assignment 行和、节点置换不变性、padding 不污染、退化空 cluster、跨图边拒绝、受控训练和带外发布信任。合成环/星图只用于验证实现，不等价于真实图分类 benchmark。

参考：[Hierarchical Graph Representation Learning with Differentiable Pooling, NeurIPS 2018](https://arxiv.org/abs/1806.08804)。其他层次池化设计可对照 [Graph U-Nets](https://arxiv.org/abs/1905.05178) 与采用 SortPooling 的 [DGCNN](https://arxiv.org/abs/1801.07829)。


In [ ]:
from __future__ import annotations

import copy
from dataclasses import dataclass
import hashlib
import json
import random
from types import MappingProxyType
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 5101
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

def canonical_digest(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def state_digest51(state: dict[str, torch.Tensor]) -> str:
    h = hashlib.sha256()
    for key in sorted(state):
        value = state[key].detach().cpu().contiguous()
        h.update(key.encode()); h.update(str(value.dtype).encode())
        h.update(str(tuple(value.shape)).encode()); h.update(value.numpy().tobytes())
    return h.hexdigest()

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1
assert torch.initial_seed() == SEED
assert state_digest51({"x": torch.tensor([1], dtype=torch.long)}) != state_digest51({"x": torch.tensor([1.0])})
assert not any(name in globals() for name in ("torch_geometric", "dgl"))


## 1. 图级切分与输入合同

类别 0 是环，类别 1 是星形，节点数 5–8。节点特征为常数、归一化 degree、局部奇偶标记；degree 由当前图拓扑计算，不直接写入 label。每类 18 张图：前 12 张 train、随后 3 张 validation、最后 3 张 test，整张图只属于一个 split。

原始无向邻接必须有限、非负、对称且对角为 0。模型内部 pooling 后允许加权自环，但输入边不允许越界、重复或自环。


In [ ]:
@dataclass(frozen=True)
class GraphItem:
    graph_id: str
    x: torch.Tensor
    edge_pairs: tuple[tuple[int, int], ...]
    label: int
    split: str

def make_graph51(label: int, index: int, split: str) -> GraphItem:
    if label not in (0, 1) or split not in {"train", "val", "test"}:
        raise ValueError("label/split 非法")
    n = 5 + index % 4
    if label == 0:
        pairs = {tuple(sorted((i, (i + 1) % n))) for i in range(n)}
    else:
        pairs = {(0, i) for i in range(1, n)}
    degree = torch.zeros(n)
    for u, v in pairs:
        degree[u] += 1; degree[v] += 1
    x = torch.stack([torch.ones(n), degree / (n - 1), 0.1 * (torch.arange(n) % 2)], dim=-1)
    return GraphItem(f"{split}-c{label}-{index:02d}", x.float(), tuple(sorted(pairs)), label, split)

def validate_graph_item51(graph: GraphItem) -> None:
    if graph.x.ndim != 2 or graph.x.shape[0] == 0 or not torch.isfinite(graph.x).all():
        raise ValueError("节点特征必须是非空有限二维张量")
    n, seen = graph.x.shape[0], set()
    for u, v in graph.edge_pairs:
        if not (0 <= u < n and 0 <= v < n) or u == v:
            raise ValueError("边越界或含自环")
        key = tuple(sorted((u, v)))
        if key in seen: raise ValueError("无向边重复")
        seen.add(key)

graphs51 = []
for label in (0, 1):
    for index in range(18):
        split = "train" if index < 12 else ("val" if index < 15 else "test")
        graphs51.append(make_graph51(label, index, split))
split_graphs51 = {s: [g for g in graphs51 if g.split == s] for s in ("train", "val", "test")}
for graph in graphs51: validate_graph_item51(graph)
assert tuple(len(split_graphs51[s]) for s in ("train", "val", "test")) == (24, 6, 6)
assert len({g.graph_id for g in graphs51}) == 36
assert set(g.label for g in split_graphs51["test"]) == {0, 1}
assert not ({g.graph_id for g in split_graphs51["train"]} & {g.graph_id for g in split_graphs51["test"]})


## 2. 多图 padded batch 与跨图边隔离

batch 张量为 `x:[B,Nmax,F]`、`adj:[B,Nmax,Nmax]`、`mask:[B,Nmax]`。padding 的 feature 和任一相关邻接必须为 0。`mask` 决定所有归一化、softmax、辅助损失和 readout 的有效范围。

若上游先构建全局稀疏边再 densify，必须检查每条边两端 `node_graph` 相同；否则跨图边会被静默写入错误样本。这里提供显式 ingestion oracle。


In [ ]:
@dataclass(frozen=True)
class DenseBatch:
    x: torch.Tensor
    adj: torch.Tensor
    mask: torch.Tensor
    labels: torch.Tensor
    graph_ids: tuple[str, ...]
    snapshot_digest: str

def validate_disjoint_edges51(edge_index: torch.Tensor, node_graph: torch.Tensor) -> None:
    if edge_index.ndim != 2 or edge_index.shape[0] != 2 or edge_index.dtype != torch.long:
        raise ValueError("edge_index 必须是 long[2,E]")
    if node_graph.ndim != 1 or node_graph.dtype != torch.long:
        raise ValueError("node_graph 必须是一维 long")
    if edge_index.numel() and (int(edge_index.min()) < 0 or int(edge_index.max()) >= node_graph.numel()):
        raise ValueError("稀疏边端点越界")
    if edge_index.numel() and not torch.equal(node_graph[edge_index[0]], node_graph[edge_index[1]]):
        raise ValueError("检测到跨图边")

def dense_snapshot_digest51(x, adj, mask, labels, graph_ids) -> str:
    tensors = {"x": x, "adj": adj, "mask": mask, "labels": labels}
    return canonical_digest({"state": state_digest51(tensors), "graph_ids": list(graph_ids)})

def pad_graphs51(items: list[GraphItem], pad_to: int | None = None) -> DenseBatch:
    if not items: raise ValueError("batch 不能为空")
    if len({g.graph_id for g in items}) != len(items): raise ValueError("graph_id 重复")
    for graph in items: validate_graph_item51(graph)
    feature_dim = items[0].x.shape[1]
    if any(g.x.shape[1] != feature_dim for g in items): raise ValueError("特征维度不一致")
    max_nodes = max(g.x.shape[0] for g in items)
    pad_to = max_nodes if pad_to is None else pad_to
    if pad_to < max_nodes: raise ValueError("pad_to 小于真实节点数")
    x = torch.zeros(len(items), pad_to, feature_dim)
    adj = torch.zeros(len(items), pad_to, pad_to)
    mask = torch.zeros(len(items), pad_to, dtype=torch.bool)
    for b, graph in enumerate(items):
        n = graph.x.shape[0]; x[b, :n] = graph.x; mask[b, :n] = True
        for u, v in graph.edge_pairs:
            adj[b, u, v] = 1.0; adj[b, v, u] = 1.0
    labels = torch.tensor([g.label for g in items], dtype=torch.long)
    ids = tuple(g.graph_id for g in items)
    digest = dense_snapshot_digest51(x, adj, mask, labels, ids)
    return DenseBatch(x, adj, mask, labels, ids, digest)

train51 = pad_graphs51(split_graphs51["train"])
val51 = pad_graphs51(split_graphs51["val"])
test51 = pad_graphs51(split_graphs51["test"])
assert train51.x.shape == (24, 8, 3) and train51.adj.shape == (24, 8, 8)
assert torch.equal(train51.adj, train51.adj.transpose(1, 2))
assert torch.count_nonzero(train51.x[~train51.mask]) == 0
assert torch.count_nonzero(train51.adj * (~train51.mask)[:, :, None]) == 0

try:
    validate_disjoint_edges51(torch.tensor([[0, 1], [1, 2]]), torch.tensor([0, 0, 1, 1]))
    raise AssertionError("跨图边未被拒绝")
except ValueError as exc:
    assert "跨图边" in str(exc)


## 3. 手写 dense GCN

对每张 padded 图，仅在有效节点加入自环，计算

$$\hat A=A+I_{valid},\qquad \tilde A=D^{-1/2}\hat A D^{-1/2},\qquad H'=\sigma(\tilde AHW).$$

mask 在归一化前后都应用；否则 padding 节点的 linear bias 或伪自环会污染后续 assignment。输入邻接允许 DiffPool 产生的非负权重和对角项，但始终要求对称、有限、padding 区域为零。对称合同使用绝对容差 `1e-5` 且 `rtol=0`（覆盖两次 float32 矩阵乘的舍入误差），避免大权重把明显绝对误差藏进相对容差；本发布版本还把单边权重上限冻结为 $10^9$。


In [ ]:
SYMMETRY_ATOL51 = 1e-5
MAX_ADJ_WEIGHT51 = 1e9

def validate_dense_tensors51(x: torch.Tensor, adj: torch.Tensor, mask: torch.Tensor) -> None:
    if x.ndim != 3 or adj.ndim != 3 or mask.ndim != 2:
        raise ValueError("x/adj/mask 维度非法")
    b, n, _ = x.shape
    if adj.shape != (b, n, n) or mask.shape != (b, n) or mask.dtype != torch.bool:
        raise ValueError("x/adj/mask shape 或 dtype 不匹配")
    if (not torch.is_floating_point(x) or not torch.is_floating_point(adj) or x.dtype != adj.dtype
            or x.device != adj.device or mask.device != x.device):
        raise ValueError("x/adj/mask 必须同设备，且 x/adj 为同 dtype 浮点张量")
    if not torch.isfinite(x).all() or not torch.isfinite(adj).all() or bool((adj < 0).any()):
        raise ValueError("特征/邻接含非有限值或负权")
    if bool((adj > MAX_ADJ_WEIGHT51).any()):
        raise ValueError(f"邻接权重超过发布上限 {MAX_ADJ_WEIGHT51:g}")
    if not torch.allclose(adj, adj.transpose(1, 2), atol=SYMMETRY_ATOL51, rtol=0.0):
        raise ValueError("邻接矩阵必须对称")
    pair_mask = mask[:, :, None] & mask[:, None, :]
    if torch.count_nonzero(x.masked_fill(mask[:, :, None], 0.0)):
        raise ValueError("padding 特征必须为零")
    if torch.count_nonzero(adj.masked_fill(pair_mask, 0.0)):
        raise ValueError("padding 邻接必须为零")
    if bool((mask.sum(1) == 0).any()): raise ValueError("batch 含空图")

def normalized_adjacency51(adj: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    dummy_x = torch.zeros(adj.shape[0], adj.shape[1], 1, device=adj.device, dtype=adj.dtype)
    validate_dense_tensors51(dummy_x, adj, mask)
    eye = torch.diag_embed(mask.to(adj.dtype))
    pair_mask = mask[:, :, None] & mask[:, None, :]
    with_self = (adj + eye) * pair_mask
    degree = with_self.sum(-1)
    inv_sqrt = torch.where(mask, degree.clamp_min(1e-12).rsqrt(), torch.zeros_like(degree))
    return with_self * inv_sqrt[:, :, None] * inv_sqrt[:, None, :]

class DenseGCN(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, activate: bool = True):
        super().__init__()
        if input_dim <= 0 or output_dim <= 0: raise ValueError("GCN 维度必须为正")
        self.linear = nn.Linear(input_dim, output_dim)
        self.activate = activate

    def forward(self, x: torch.Tensor, adj: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        validate_dense_tensors51(x, adj, mask)
        support = self.linear(x)
        out = torch.bmm(normalized_adjacency51(adj, mask), support)
        if self.activate: out = F.relu(out)
        if not torch.isfinite(out).all(): raise ValueError("GCN 输出含非有限值")
        return out * mask[:, :, None]

gcn_probe51 = DenseGCN(3, 5)(train51.x[:2], train51.adj[:2], train51.mask[:2])
assert gcn_probe51.shape == (2, 8, 5)
assert torch.count_nonzero(gcn_probe51[~train51.mask[:2]]) == 0

bad_adj51 = train51.adj[:1].clone(); bad_adj51[0, 0, 2] = 0.5
try:
    DenseGCN(3, 4)(train51.x[:1], bad_adj51, train51.mask[:1])
    raise AssertionError("非对称邻接未被拒绝")
except ValueError as exc:
    assert "对称" in str(exc)

large_asym51 = torch.tensor([[[0.0, 100000000.0], [100000512.0, 0.0]]])
try:
    validate_dense_tensors51(torch.ones(1, 2, 1), large_asym51, torch.tensor([[True, True]]))
    raise AssertionError("大权重下绝对差 512 的非对称邻接被默认 rtol 放过")
except ValueError as exc:
    assert "对称" in str(exc)
assert not torch.allclose(large_asym51, large_asym51.transpose(1, 2), atol=SYMMETRY_ATOL51, rtol=0.0)


## 4. Assignment、池化公式与辅助损失

assignment network 输出 `logits:[B,N,K]`，只对有效节点行做 softmax：有效行和为 1，padding 行严格为 0。随后

$$X' = S^T Z,\qquad A'=S^TAS.$$

link 辅助项逼近邻接：$\|A-SS^T\|_F^2$，只统计有效节点对；entropy 项 $-\sum_k S_{ik}\log S_{ik}$ 抑制完全模糊的分配。link 在平方前按每张图的最大绝对误差缩放到安全范围，求均值后再恢复量纲；配合邻接权重上限与终端有限性检查，避免有限的大权重在 float32 平方时静默变成 `inf`。二者是正则项，不应混入 test 标签。

若某 cluster 的总 assignment mass 为 0，它被标记为无效，输出行/列清零，不能除零或产生 NaN。


In [ ]:
def masked_assignment_softmax51(logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    if logits.ndim != 3 or mask.shape != logits.shape[:2] or mask.dtype != torch.bool:
        raise ValueError("assignment logits/mask 合同不匹配")
    if logits.shape[-1] <= 0 or not torch.isfinite(logits).all():
        raise ValueError("cluster 数或 logits 数值非法")
    out = torch.zeros_like(logits)
    out[mask] = torch.softmax(logits[mask], dim=-1)
    return out

def diffpool_tensors51(z: torch.Tensor, adj: torch.Tensor, mask: torch.Tensor,
                       assignment: torch.Tensor, eps: float = 1e-8):
    validate_dense_tensors51(z, adj, mask)
    if (not isinstance(eps, (int, float)) or isinstance(eps, bool)
            or not np.isfinite(float(eps)) or float(eps) <= 0):
        raise ValueError("eps 必须是有限正数")
    if assignment.shape[:2] != mask.shape or assignment.ndim != 3 or not torch.isfinite(assignment).all():
        raise ValueError("assignment shape/数值非法")
    if (not torch.is_floating_point(assignment) or assignment.dtype != z.dtype
            or assignment.device != z.device):
        raise ValueError("assignment dtype/device 必须与节点表示一致")
    if bool((assignment < 0).any()): raise ValueError("assignment 不能为负")
    row_sum = assignment.sum(-1)
    if not torch.allclose(row_sum[mask], torch.ones_like(row_sum[mask]), atol=1e-5):
        raise ValueError("有效 assignment 行和必须为 1")
    if torch.count_nonzero(assignment[~mask]): raise ValueError("padding assignment 必须为零")
    pair_mask = mask[:, :, None] & mask[:, None, :]
    z_clean = z * mask[:, :, None]
    adj_clean = adj * pair_mask
    pooled_x = torch.bmm(assignment.transpose(1, 2), z_clean)
    pooled_adj = torch.bmm(torch.bmm(assignment.transpose(1, 2), adj_clean), assignment)
    cluster_mass = assignment.sum(1)
    cluster_mask = cluster_mass > eps
    cluster_pair = cluster_mask[:, :, None] & cluster_mask[:, None, :]
    pooled_x = pooled_x * cluster_mask[:, :, None]
    pooled_adj = pooled_adj * cluster_pair
    if not torch.isfinite(pooled_x).all() or not torch.isfinite(pooled_adj).all():
        raise ValueError("DiffPool pooled tensor 含非有限值")
    reconstructed = torch.bmm(assignment, assignment.transpose(1, 2))
    error = adj_clean - reconstructed
    # 每图先缩放到绝对值不超过约 1 再平方，最后恢复量纲；结合 1e9 输入上限可避免 float32 overflow。
    error_scale = error.detach().abs().amax((1, 2), keepdim=True).clamp_min(1.0)
    scaled_squared = (error / error_scale).square() * pair_mask
    link_per_graph = (scaled_squared.sum((1, 2)) / pair_mask.sum((1, 2)).clamp_min(1)) * error_scale.flatten().square()
    entropy_node = -(assignment.clamp_min(1e-12).log() * assignment).sum(-1)
    entropy_per_graph = (entropy_node * mask).sum(1) / mask.sum(1).clamp_min(1)
    link_loss = link_per_graph.mean()
    entropy_loss = entropy_per_graph.mean()
    if not torch.isfinite(link_loss) or not torch.isfinite(entropy_loss):
        raise ValueError("DiffPool 辅助损失含非有限值")
    return pooled_x, pooled_adj, cluster_mask, link_loss, entropy_loss

logits_probe51 = torch.tensor([[[2.0, 0.0], [0.0, 2.0], [99.0, -99.0]]])
mask_probe51 = torch.tensor([[True, True, False]])
assign_probe51 = masked_assignment_softmax51(logits_probe51, mask_probe51)
assert torch.allclose(assign_probe51[0, :2].sum(-1), torch.ones(2))
assert torch.equal(assign_probe51[0, 2], torch.zeros(2))

z_deg51 = torch.tensor([[[1.0], [2.0], [0.0]]])
adj_deg51 = torch.tensor([[[0., 1., 0.], [1., 0., 0.], [0., 0., 0.]]])
mask_deg51 = torch.tensor([[True, True, False]])
assign_deg51 = torch.tensor([[[1., 0.], [1., 0.], [0., 0.]]])
px_deg51, pa_deg51, pm_deg51, link_deg51, ent_deg51 = diffpool_tensors51(
    z_deg51, adj_deg51, mask_deg51, assign_deg51)
assert pm_deg51.tolist() == [[True, False]]
assert torch.allclose(px_deg51[0, 0], torch.tensor([3.0]))
assert torch.allclose(pa_deg51[0, 0, 0], torch.tensor(2.0))
assert torch.equal(px_deg51[0, 1], torch.zeros(1))
assert torch.equal(pa_deg51[0, 1], torch.zeros(2)) and torch.isfinite(pa_deg51).all()
assert torch.allclose(link_deg51, torch.tensor(0.5)) and float(ent_deg51) == 0.0

extreme_adj51 = torch.tensor([[[0.0, torch.finfo(torch.float32).max / 2],
                               [torch.finfo(torch.float32).max / 2, 0.0]]])
try:
    diffpool_tensors51(torch.ones(1, 2, 1), extreme_adj51, torch.tensor([[True, True]]),
                       torch.tensor([[[1.0, 0.0], [0.0, 1.0]]]))
    raise AssertionError("可能令平方 link loss 溢出的极值邻接未 fail-closed")
except ValueError as exc:
    assert "上限" in str(exc)
safe_large51 = torch.tensor([[[0.0, 1e8], [1e8, 0.0]]])
safe_result51 = diffpool_tensors51(torch.ones(1, 2, 1), safe_large51, torch.tensor([[True, True]]),
                                    torch.tensor([[[1.0, 0.0], [0.0, 1.0]]]))
assert torch.isfinite(safe_result51[1]).all() and torch.isfinite(safe_result51[3])


## 5. 手写 DiffPool layer

一个 layer 有两条独立 GCN 支路：embedding GNN 产生 $Z$，assignment GNN 产生 $S$。共享参数会无意限制两者表达。`forward` 返回 pooled tensors、mask、两个辅助损失和 assignment，便于审计而不是把关键中间量藏起来。


In [ ]:
class DiffPoolLayer(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, num_clusters: int):
        super().__init__()
        if num_clusters <= 0: raise ValueError("num_clusters 必须为正")
        self.embed_gcn1 = DenseGCN(input_dim, hidden_dim)
        self.embed_gcn2 = DenseGCN(hidden_dim, output_dim)
        self.assign_gcn = DenseGCN(input_dim, hidden_dim)
        self.assign_linear = nn.Linear(hidden_dim, num_clusters)

    def forward(self, x: torch.Tensor, adj: torch.Tensor, mask: torch.Tensor):
        z = self.embed_gcn2(self.embed_gcn1(x, adj, mask), adj, mask)
        assign_hidden = self.assign_gcn(x, adj, mask)
        assign_logits = self.assign_linear(assign_hidden)
        assignment = masked_assignment_softmax51(assign_logits, mask)
        pooled_x, pooled_adj, pooled_mask, link_loss, entropy_loss = diffpool_tensors51(
            z, adj, mask, assignment)
        return pooled_x, pooled_adj, pooled_mask, link_loss, entropy_loss, assignment

layer_probe51 = DiffPoolLayer(3, 8, 6, 4)
lp_x51, lp_a51, lp_m51, lp_link51, lp_ent51, lp_s51 = layer_probe51(
    train51.x[:2], train51.adj[:2], train51.mask[:2])
assert lp_x51.shape == (2, 4, 6) and lp_a51.shape == (2, 4, 4)
assert torch.allclose(lp_a51, lp_a51.transpose(1, 2), atol=1e-6)
assert torch.allclose(lp_s51.sum(-1)[train51.mask[:2]], torch.ones(int(train51.mask[:2].sum())), atol=1e-6)
assert float(lp_link51) >= 0 and float(lp_ent51) >= 0
assert layer_probe51.embed_gcn1.linear.weight.data_ptr() != layer_probe51.assign_gcn.linear.weight.data_ptr()


## 6. 两层 hierarchical pooling 与图级读出

第一层把最多 8 个节点软聚合为 4 个 cluster，第二层再聚合为 2 个 super-cluster。最终对有效 super-cluster 做 masked mean，得到 `[B,D]`，分类头输出 `[B,2]`。

pool 后邻接通常是稠密加权图，第二层不能再用“原始邻接必须二值/零对角”的校验规则；但对称性、非负性和 padding 清零仍是硬合同。


In [ ]:
class HierarchicalDiffPool(nn.Module):
    def __init__(self, input_dim: int = 3, hidden_dim: int = 16,
                 embed_dim: int = 12, clusters1: int = 4, clusters2: int = 2,
                 num_classes: int = 2):
        super().__init__()
        self.pool1 = DiffPoolLayer(input_dim, hidden_dim, embed_dim, clusters1)
        self.pool2 = DiffPoolLayer(embed_dim, hidden_dim, embed_dim, clusters2)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, batch: DenseBatch, return_debug: bool = False):
        x1, a1, m1, link1, ent1, s1 = self.pool1(batch.x, batch.adj, batch.mask)
        x2, a2, m2, link2, ent2, s2 = self.pool2(x1, a1, m1)
        graph_repr = (x2 * m2[:, :, None]).sum(1) / m2.sum(1, keepdim=True).clamp_min(1)
        logits = self.classifier(graph_repr)
        aux = {"link": link1 + link2, "entropy": ent1 + ent2}
        if (not torch.isfinite(graph_repr).all() or not torch.isfinite(logits).all()
                or not torch.isfinite(aux["link"]) or not torch.isfinite(aux["entropy"])):
            raise ValueError("层次 DiffPool 输出含非有限值")
        debug = {"s1": s1, "s2": s2, "mask1": m1, "mask2": m2, "adj2": a2}
        return (logits, aux, debug) if return_debug else (logits, aux)

torch.manual_seed(5102)
model_probe51 = HierarchicalDiffPool()
probe_logits51, probe_aux51, probe_debug51 = model_probe51(train51, return_debug=True)
assert probe_logits51.shape == (24, 2)
assert probe_debug51["s1"].shape == (24, 8, 4)
assert probe_debug51["s2"].shape == (24, 4, 2)
assert torch.allclose(probe_debug51["adj2"], probe_debug51["adj2"].transpose(1, 2), atol=1e-5)
assert torch.isfinite(probe_aux51["link"] + probe_aux51["entropy"])


## 7. 节点置换不变性与 padding 不污染

图分类必须对节点编号置换不变：同时置换 $X$ 的节点轴、$A$ 的两个节点轴和 mask，logits 应不变。只扩大 padding 长度也不能改变有效图输出。两项检查能捕捉“只 mask key、不 mask query/归一化/readout”等常见错误。


In [ ]:
model_probe51.eval()
single51 = pad_graphs51([split_graphs51["test"][0]], pad_to=8)
n51 = int(single51.mask[0].sum())
valid_perm51 = torch.arange(n51 - 1, -1, -1)
perm51 = torch.cat([valid_perm51, torch.arange(n51, single51.x.shape[1])])
permuted51 = DenseBatch(
    single51.x[:, perm51], single51.adj[:, perm51][:, :, perm51], single51.mask[:, perm51],
    single51.labels, single51.graph_ids, single51.snapshot_digest,
)
with torch.no_grad():
    original_logit51 = model_probe51(single51)[0]
    permuted_logit51 = model_probe51(permuted51)[0]
assert torch.allclose(original_logit51, permuted_logit51, atol=1e-5)

pair_pad8_51 = pad_graphs51(split_graphs51["test"][:2], pad_to=8)
pair_pad11_51 = pad_graphs51(split_graphs51["test"][:2], pad_to=11)
with torch.no_grad():
    logits_pad8_51 = model_probe51(pair_pad8_51)[0]
    logits_pad11_51 = model_probe51(pair_pad11_51)[0]
assert torch.allclose(logits_pad8_51, logits_pad11_51, atol=1e-5)
assert torch.count_nonzero(pair_pad11_51.x[:, 8:]) == 0
assert pair_pad8_51.snapshot_digest != pair_pad11_51.snapshot_digest


## 8. 受控训练与模型选择

目标为 `cross_entropy + 0.03*link_loss + 0.002*entropy`。训练仅访问 train batch；每 5 步在 validation 上选择 checkpoint；加载最佳参数后才评 test。辅助项权重是 recipe 的一部分，变更后应重新发布而非静默沿用旧阈值。


In [ ]:
def accuracy51(model, batch):
    model.eval()
    with torch.no_grad(): pred = model(batch)[0].argmax(-1)
    return float((pred == batch.labels).float().mean())

torch.manual_seed(5103)
model51 = HierarchicalDiffPool()
optimizer51 = torch.optim.Adam(model51.parameters(), lr=0.025)
best_val51, best_state51, best_step51, ce_trace51 = -1.0, None, None, []
for step in range(101):
    model51.train(); optimizer51.zero_grad()
    logits, aux = model51(train51)
    ce = F.cross_entropy(logits, train51.labels)
    loss = ce + 0.03 * aux["link"] + 0.002 * aux["entropy"]
    loss.backward(); torch.nn.utils.clip_grad_norm_(model51.parameters(), 5.0); optimizer51.step()
    ce_trace51.append(float(ce.detach()))
    if step % 5 == 0:
        val_score = accuracy51(model51, val51)
        if val_score > best_val51:
            best_val51 = val_score; best_state51 = copy.deepcopy(model51.state_dict()); best_step51 = step
model51.load_state_dict(best_state51)

assert len(ce_trace51) == 101
assert min(ce_trace51) < ce_trace51[0] * 0.40
assert ce_trace51[-1] < ce_trace51[0] * 0.75
assert best_val51 == 1.0
assert best_step51 is not None and best_step51 % 5 == 0
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model51.parameters())
assert all(torch.isfinite(p).all() for p in model51.parameters())


## 9. 冻结测试与中间合同复核

最终同时检查 train/validation/test accuracy，以及 test assignment 行和和 pooled adjacency 对称性。受控图由固定规则生成，因此 100% 只说明实现学会该规则；真实任务仍需跨来源、跨时间外推与置信区间。


In [ ]:
train_acc51 = accuracy51(model51, train51)
val_acc51 = accuracy51(model51, val51)
test_acc51 = accuracy51(model51, test51)
model51.eval()
with torch.no_grad(): final_logits51, final_aux51, final_debug51 = model51(test51, return_debug=True)
assert train_acc51 == 1.0 and val_acc51 == 1.0 and test_acc51 == 1.0
assert final_logits51.shape == (6, 2)
assert torch.allclose(final_debug51["s1"].sum(-1)[test51.mask], torch.ones(int(test51.mask.sum())), atol=1e-5)
assert torch.allclose(final_debug51["adj2"], final_debug51["adj2"].transpose(1, 2), atol=1e-5)
assert float(final_aux51["link"]) >= 0 and float(final_aux51["entropy"]) >= 0


## 10. 制品合同与包外 publisher registry

manifest 绑定两层 cluster 数、图/schema 规则、train/validation/test snapshot、graph ID split、padding/归一化算法和完整训练 recipe。canonical state digest 逐项覆盖参数 key、dtype、shape 与 bytes。loader 返回 `PublishedDiffPool` evaluator；它根据完整 graph ID 序列确定登记 split，并重算 x/adj/mask/labels/graph_ids 摘要，不能靠调用方填写的 `snapshot_digest` 自证。

攻击者能替换整个 package 并重算内部 hash，故内部 hash 只能检测传输损坏。loader 还检查 package 外只读 publisher registry；伪造包无法自行更新该信任锚。这个发布 evaluator 只复核登记的离线快照；在线新图分类应使用另一个明确版本化的 schema/feature 服务 API，而不是携带伪 snapshot 字段。


In [ ]:
RELEASE51 = "diffpool-demo-51/v1"
CONFIG51 = {"input_dim": 3, "hidden_dim": 16, "embed_dim": 12,
            "clusters1": 4, "clusters2": 2, "num_classes": 2}
MANIFEST51 = {
    "config": CONFIG51,
    "snapshots": {"train": train51.snapshot_digest, "val": val51.snapshot_digest, "test": test51.snapshot_digest},
    "split": {s: [g.graph_id for g in split_graphs51[s]] for s in ("train", "val", "test")},
    "runtime": {"policy": "registered_snapshot_evaluator_only",
                "digest_fields": ["x", "adj", "mask", "labels", "graph_ids"]},
    "schema": {"feature_dim": 3, "undirected": True, "raw_diagonal": 0,
               "cross_graph_edges": "reject", "symmetry_atol": SYMMETRY_ATOL51,
               "symmetry_rtol": 0.0, "max_adj_weight": MAX_ADJ_WEIGHT51,
               "link_stabilization": "per_graph_max_abs_error_scale"},
    "recipe": {"seed": SEED, "steps": 101, "optimizer": "Adam", "lr": 0.025,
               "loss": "CE+0.03*link+0.002*entropy", "normalization": "D^-1/2(A+I_valid)D^-1/2",
               "grad_clip_norm": 5.0, "validation_interval": 5,
               "selection": "best_validation_accuracy", "tie_break": "first_strict_improvement",
               "checkpoint_step": best_step51},
}
state51 = {k: v.detach().cpu().clone() for k, v in model51.state_dict().items()}
package51 = {"release_id": RELEASE51, "manifest": copy.deepcopy(MANIFEST51), "state": state51}
package51["state_digest"] = state_digest51(state51)
package51["package_digest"] = canonical_digest({"release_id": RELEASE51, "manifest": package51["manifest"],
                                                  "state_digest": package51["state_digest"]})
_PUBLISHER_REGISTRY51 = MappingProxyType({RELEASE51: package51["package_digest"]})

def validate_registered_batch51(batch: DenseBatch, manifest: dict) -> str:
    if not isinstance(batch, DenseBatch): raise ValueError("输入必须是 DenseBatch")
    validate_dense_tensors51(batch.x, batch.adj, batch.mask)
    b = batch.x.shape[0]
    if batch.labels.shape != (b,) or batch.labels.dtype != torch.long or batch.labels.device != batch.x.device:
        raise ValueError("DenseBatch labels 合同不匹配")
    if (len(batch.graph_ids) != b or len(set(batch.graph_ids)) != b
            or any(not isinstance(g, str) or not g for g in batch.graph_ids)):
        raise ValueError("DenseBatch graph_ids 合同不匹配")
    matching = [split for split, ids in manifest["split"].items() if tuple(ids) == tuple(batch.graph_ids)]
    if len(matching) != 1: raise ValueError("graph_ids 未唯一匹配发布 split")
    split = matching[0]
    actual = dense_snapshot_digest51(batch.x, batch.adj, batch.mask, batch.labels, batch.graph_ids)
    if not isinstance(batch.snapshot_digest, str) or batch.snapshot_digest != actual:
        raise ValueError("snapshot 自带摘要与 tensor/metadata 不一致")
    if actual != manifest["snapshots"][split]:
        raise ValueError("snapshot 未在发布 manifest 登记")
    return split

class PublishedDiffPool(nn.Module):
    def __init__(self, model: HierarchicalDiffPool, manifest: dict):
        super().__init__(); self.model = model; self._manifest = copy.deepcopy(manifest)

    def forward(self, batch: DenseBatch, return_debug: bool = False):
        validate_registered_batch51(batch, self._manifest)
        return self.model(batch, return_debug=return_debug)

def load_published_diffpool(package: dict) -> PublishedDiffPool:
    if set(package) != {"release_id", "manifest", "state", "state_digest", "package_digest"}:
        raise ValueError("package 字段集合非法")
    release_id = package["release_id"]
    if release_id not in _PUBLISHER_REGISTRY51: raise ValueError("未知 release")
    actual_state = state_digest51(package["state"])
    actual_package = canonical_digest({"release_id": release_id, "manifest": package["manifest"],
                                       "state_digest": actual_state})
    if actual_state != package["state_digest"] or actual_package != package["package_digest"]:
        raise ValueError("包内摘要不匹配")
    if actual_package != _PUBLISHER_REGISTRY51[release_id]:
        raise ValueError("publisher registry 信任锚不匹配")
    if package["manifest"] != MANIFEST51:
        raise ValueError("config/snapshot/split/schema/recipe 合同不匹配")
    model = HierarchicalDiffPool(**package["manifest"]["config"])
    model.load_state_dict(package["state"], strict=True); model.eval()
    restored = PublishedDiffPool(model, package["manifest"]); restored.eval()
    return restored

restored51 = load_published_diffpool(package51)
assert torch.allclose(restored51(test51)[0], model51(test51)[0], atol=1e-7)
assert isinstance(restored51, PublishedDiffPool)
assert package51["manifest"]["snapshots"]["train"] == train51.snapshot_digest

fake_digest51 = DenseBatch(test51.x, test51.adj, test51.mask, test51.labels, test51.graph_ids, "attacker-digest")
fake_labels_draft51 = DenseBatch(test51.x, test51.adj, test51.mask, 1 - test51.labels, test51.graph_ids, "pending")
fake_labels51 = DenseBatch(fake_labels_draft51.x, fake_labels_draft51.adj, fake_labels_draft51.mask,
                           fake_labels_draft51.labels, fake_labels_draft51.graph_ids,
                           dense_snapshot_digest51(fake_labels_draft51.x, fake_labels_draft51.adj,
                                                   fake_labels_draft51.mask, fake_labels_draft51.labels,
                                                   fake_labels_draft51.graph_ids))
snapshot_rejections51 = {}
for attack_name, attacked in {"digest": fake_digest51, "labels": fake_labels51}.items():
    try:
        restored51(attacked)
        raise AssertionError(f"伪造 snapshot {attack_name} 被接受")
    except ValueError as exc:
        snapshot_rejections51[attack_name] = str(exc)
assert set(snapshot_rejections51) == {"digest", "labels"}

forged51 = copy.deepcopy(package51)
forged51["manifest"]["recipe"]["loss"] = "attacker-controlled"
first_key51 = next(iter(forged51["state"]))
forged51["state"][first_key51] = forged51["state"][first_key51] + 0.02
forged51["state_digest"] = state_digest51(forged51["state"])
forged51["package_digest"] = canonical_digest({"release_id": RELEASE51, "manifest": forged51["manifest"],
                                                 "state_digest": forged51["state_digest"]})
try:
    load_published_diffpool(forged51)
    raise AssertionError("整体替换 manifest 并重算内部摘要后仍被接受")
except ValueError as exc:
    assert "registry" in str(exc)


## 11. 复杂度、失败模式与生产差距

- **复杂度**：dense GCN 为 $O(BN^2D)$；一次 pooling 的 $S^TAS$ 约 $O(BN^2K+BNK^2)$。大图必须稀疏化、分区或采样，不能直接 padding 到全局最大节点数。
- **失败模式**：padding 行参与 softmax；只置换 feature 没同步 adjacency；非对称邻接静默进入；空 cluster 除零；跨图边污染；把 test 图参与 assignment 预训练。
- **模型风险**：cluster 缺少可解释语义，entropy/link 权重不当会导致全部节点挤进一个 cluster 或均匀分配。应监控 cluster mass、熵、稳定性和 OOD 图规模。
- **发布差距**：真实系统还需特征版本、图快照水位、稀疏算子 ABI、数值容差、签名/KMS、灰度和回滚。


In [ ]:
assert test_acc51 == 1.0
assert package51["package_digest"] == _PUBLISHER_REGISTRY51[RELEASE51]
assert set(MANIFEST51["snapshots"]) == {"train", "val", "test"}
assert final_debug51["mask2"].shape == (6, 2)
assert all(parameter.device.type == "cpu" for parameter in restored51.parameters())
